# Chạy pipeline YouTube — Stage 1–6

Notebook này là entry point thay cho `python -m ypipe`: nạp Takeout, gọi enrichment qua `ypipe.enrich`, lưu DuckDB/Parquet, tạo đặc trưng, phân cụm và xuất báo cáo HTML. Các định nghĩa `store`, `report` và điều phối pipeline nằm ngay trong notebook; connection DuckDB luôn đóng trong `finally`.

**Cách chạy:** đặt file Takeout `.json`/`.zip` vào `inbox/`, cấu hình API key trong `.env` nếu dùng API thật, kiểm tra giới hạn ở cell tham số, rồi chọn **Restart Kernel and Run All**. Đặt `ENRICH_API=False` để chạy offline; `YTPIPE_MOCK_API` vẫn được `ypipe.enrich` hỗ trợ. Không chạy đồng thời notebook phân tích khác và pipeline trên cùng database.

Pipeline ghi bền vào `data/warehouse.duckdb` và snapshot Parquet vào `data/parquet/`. Schema được migrate tại chỗ để giữ database cũ.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ypipe').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from ypipe import cluster, config, enrich, features, ingest, nlp

SEED = 42
np.random.seed(SEED)
config.DATA.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_ROOT} | DuckDB: {config.DB_PATH}")

Project: d:\project\bigdata | DuckDB: D:\project\bigdata\data\warehouse.duckdb


In [2]:
# Tham số chạy: giới hạn tổng số batch API video + channel trong lượt này.
RUN_PIPELINE = True
ENRICH_API = True
MAX_API_CALLS = 10000
FORCE = False

if MAX_API_CALLS is not None and MAX_API_CALLS < 0:
    raise ValueError('MAX_API_CALLS phải >= 0')
print(f"RUN_PIPELINE={RUN_PIPELINE} | ENRICH_API={ENRICH_API} | MAX_API_CALLS={MAX_API_CALLS} | FORCE={FORCE}")

RUN_PIPELINE=True | ENRICH_API=True | MAX_API_CALLS=10000 | FORCE=False


In [3]:
"""Stage 4: Storage - DuckDB (1 file, miễn phí, không cần server). Có thể thay bằng PostgreSQL ở đây."""
import duckdb
import pandas as pd
from ypipe import config

SCHEMA = """
CREATE TABLE IF NOT EXISTS watch_raw(
  user_id VARCHAR, watched_at TIMESTAMP, video_id VARCHAR, title VARCHAR,
  channel_name VARCHAR, channel_id VARCHAR, product VARCHAR);
CREATE TABLE IF NOT EXISTS ingested_files(
  sha256 VARCHAR PRIMARY KEY, user_id VARCHAR, filename VARCHAR,
  rows_kept INTEGER, rows_new INTEGER, ingested_at TIMESTAMP);
CREATE TABLE IF NOT EXISTS video_metadata(
  video_id VARCHAR PRIMARY KEY, status VARCHAR, title VARCHAR, description VARCHAR,
  channel_id VARCHAR, category_id INTEGER, category_name VARCHAR, published_at TIMESTAMP,
  duration_sec DOUBLE, view_count BIGINT, like_count BIGINT, comment_count BIGINT,
  tags VARCHAR, default_language VARCHAR, fetched_at TIMESTAMP,
  made_for_kids BOOLEAN, status_fetched_at TIMESTAMP);
CREATE TABLE IF NOT EXISTS channel_metadata(
  channel_id VARCHAR PRIMARY KEY, subscriber_count BIGINT, video_count BIGINT,
  fetched_at TIMESTAMP);
CREATE TABLE IF NOT EXISTS video_comments(
  comment_id VARCHAR PRIMARY KEY, video_id VARCHAR, author VARCHAR,
  text VARCHAR, published_at TIMESTAMP, like_count BIGINT);
"""


def connect():
    config.DATA.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(str(config.DB_PATH))
    con.execute(SCHEMA)
    # DuckDB files created by earlier versions already have video_metadata.
    # Add new fields in place so existing warehouses remain usable.
    columns = {row[0] for row in con.execute("DESCRIBE video_metadata").fetchall()}
    if "made_for_kids" not in columns:
        con.execute("ALTER TABLE video_metadata ADD COLUMN made_for_kids BOOLEAN")
    if "status_fetched_at" not in columns:
        con.execute("ALTER TABLE video_metadata ADD COLUMN status_fetched_at TIMESTAMP")
    return con


def insert_df(con, table, df):
    cols = [r[0] for r in con.execute(f"DESCRIBE {table}").fetchall()]
    df = df[cols]
    con.register("_df", df)
    con.execute(f"INSERT INTO {table} SELECT * FROM _df")
    con.unregister("_df")


def replace_table(con, table, df):
    con.register("_df", df)
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM _df")
    con.unregister("_df")
    config.PARQUET.mkdir(parents=True, exist_ok=True)
    df.to_parquet(config.PARQUET / f"{table}.parquet", index=False)


def export_parquet(con, table):
    """Xuất snapshot Parquet của bảng đã được cập nhật theo từng lô."""
    df = con.execute(f"SELECT * FROM {table}").df()
    config.PARQUET.mkdir(parents=True, exist_ok=True)
    df.to_parquet(config.PARQUET / f"{table}.parquet", index=False)


def read(con, sql):
    return con.execute(sql).df()


from types import SimpleNamespace
store = SimpleNamespace(connect=connect, insert_df=insert_df, replace_table=replace_table,
                        export_parquet=export_parquet, read=read)
print('Store DuckDB/Parquet đã sẵn sàng; schema cũ được migrate khi connect().')
# API lưu trữ cục bộ dùng bởi hàm pipeline bên dưới.
from types import SimpleNamespace
store = SimpleNamespace(connect=connect, insert_df=insert_df, replace_table=replace_table, export_parquet=export_parquet, read=read)
ingest.store = store
enrich.store = store


Store DuckDB/Parquet đã sẵn sàng; schema cũ được migrate khi connect().


In [4]:
"""Stage 6: Visualization - báo cáo HTML tự chứa (ảnh nhúng base64) cho từng người và cho cả nhóm."""
import base64
import html
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ypipe import config, features as F
from ypipe.cluster import LABEL

C1, C2, GRID = "#2563eb", "#f59e0b", "#e5e7eb"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#9ca3af", "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True})
CSS = """body{font-family:system-ui,Segoe UI,Roboto,sans-serif;max-width:1000px;margin:24px auto;padding:0 16px;color:#111827}
h1{margin-bottom:0}.sub{color:#6b7280;margin-top:4px}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:16px 0}
.card{background:#f3f4f6;border-radius:10px;padding:10px 14px}.card b{font-size:22px;display:block}.card span{color:#6b7280;font-size:12px}
img{max-width:100%}table{border-collapse:collapse;width:100%;font-size:13px}td,th{border-bottom:1px solid #e5e7eb;padding:5px 8px;text-align:left}
.note{background:#fffbeb;border-left:4px solid #f59e0b;padding:8px 12px;font-size:13px;margin:14px 0}li{margin:5px 0}h2{margin-top:30px}"""


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    return '<img src="data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode() + '">'


def _pct(x):
    return f"{x * 100:.0f}%"


def _card(v, label):
    return f'<div class="card"><b>{v}</b><span>{label}</span></div>'


def _rank_note(uf, uid, col, label):
    if len(uf) < config.MIN_USERS_CLUSTER or uf[col].isna().all() or pd.isna(uf.loc[uf.user_id == uid, col].iloc[0]):
        return ""
    v = uf.loc[uf.user_id == uid, col].iloc[0]
    pr = (uf[col].dropna() < v).mean()
    return f"<tr><td>{label}</td><td>{v:.2f}</td><td>cao hơn {pr * 100:.0f}% nhóm</td></tr>"


def insights(f, topics_txt):
    out = []
    out.append(f"Dữ liệu gồm <b>{f.n_events:,}</b> lượt xem trong <b>{f.span_days}</b> ngày "
               f"({f.first_date:%d/%m/%Y} → {f.last_date:%d/%m/%Y}), hoạt động {_pct(f.active_day_ratio)} số ngày, "
               f"trung bình <b>{f.videos_per_active_day:.1f}</b> video/ngày hoạt động.")
    out.append(f"Giờ xem nhiều nhất: <b>{f.peak_hour}h</b>. Cơ cấu: sáng {_pct(f.dp_sang)}, chiều {_pct(f.dp_chieu)}, "
               f"tối {_pct(f.dp_toi)}, đêm (22h–5h) {_pct(f.dp_dem)}.")
    if not pd.isna(f.weekend_boost):
        d = "nhiều hơn" if f.weekend_boost > 1.1 else "ít hơn" if f.weekend_boost < 0.9 else "tương đương"
        out.append(f"Cuối tuần xem {d} ngày thường (×{f.weekend_boost:.2f} số video/ngày).")
    out.append(f"Ước tính <b>{f.n_sessions:,}</b> phiên (ngắt phiên nếu im lặng >{config.SESSION_GAP_MIN} phút): "
               f"{f.avg_videos_per_session:.1f} video/phiên, phiên dài nhất {f.max_videos_per_session} video, "
               f"{_pct(f.binge_session_ratio)} phiên có ≥8 video.")
    out.append(f"Top 5 kênh chiếm {_pct(f.channel_top5_share)} lượt xem, top 1 chiếm {_pct(f.channel_top1_share)}; "
               f"{_pct(f.repeat_ratio)} lượt xem là xem lại video đã xem.")
    if not pd.isna(f.shorts_ratio):
        out.append(f"Shorts (≤{config.SHORTS_MAX_SEC}s) chiếm {_pct(f.shorts_ratio)}, video dài ≥20 phút chiếm {_pct(f.long_ratio)}; "
                   f"thể loại nhiều nhất: <b>{html.escape(str(f.top_category))}</b> ({_pct(f.top_category_share)}).")
    elif topics_txt:
        out.append(f"Chủ đề nổi bật theo NLP trên title: {html.escape(topics_txt)}.")
    if not pd.isna(f.topic_drift):
        lv = "đáng kể" if f.topic_drift > 0.3 else "nhẹ" if f.topic_drift > 0.15 else "rất ít"
        out.append(f"Sở thích thay đổi {lv} giữa nửa đầu và nửa sau giai đoạn "
                   f"(JSD={f.topic_drift:.2f}, tính trên {'thể loại' if f.drift_basis == 'category' else 'kênh'}).")
    if hasattr(f, "comment_positive_share") and pd.notna(f.comment_positive_share):
        out.append(f"Tín hiệu bình luận trên các video đã xem: tích cực {_pct(f.comment_positive_share)}, "
                   f"tiêu cực {_pct(f.comment_negative_share)} (không phải cảm xúc trực tiếp của người dùng).")
    return out


def _topic_name(topic_id, topic_lbl):
    if topic_lbl is None or topic_lbl.empty:
        return f"Chủ đề {topic_id}"
    labels = topic_lbl.set_index("topic_id")["label"]
    return str(labels.get(topic_id, f"Chủ đề {topic_id}"))


def _transition_text(row, topic_lbl):
    return (f"{_topic_name(row.source_topic_id, topic_lbl)} → "
            f"{_topic_name(row.target_topic_id, topic_lbl)}")


def _session_sensitivity_note(sensitivity, uid):
    """Trả về bảng và kết luận thận trọng về ngưỡng phiên cho một người dùng."""
    s = sensitivity[sensitivity["user_id"] == uid].sort_values("gap_minutes")
    if s.empty:
        return ""
    metrics = ["n_sessions", "avg_videos_per_session"]
    base = s[s["gap_minutes"] == config.SESSION_GAP_MIN]
    baseline = base.iloc[0] if len(base) else s.iloc[len(s) // 2]
    rel_change = max(abs(s[m] - baseline[m]).max() / max(float(baseline[m]), 1) for m in metrics)
    verdict = "nhạy với ngưỡng" if rel_change >= 0.25 else "tương đối ổn định theo ngưỡng"
    rows = "".join(
        f"<tr><td>{r.gap_minutes}</td><td>{r.n_sessions:,}</td><td>{r.avg_videos_per_session:.1f}</td>"
        f"<td>{_pct(r.binge_session_ratio)}</td></tr>" for r in s.itertuples())
    return (f"<h2>Kiểm định độ nhạy phiên xem</h2><p>Ở người này, các chỉ số phiên <b>{verdict}</b> "
            f"khi đổi ngưỡng im lặng 15/30/60 phút (biến thiên lớn nhất {rel_change:.0%} so với mốc 30 phút).</p>"
            f"<table><tr><th>Ngưỡng (phút)</th><th>Số phiên</th><th>Video/phiên</th><th>Phiên ≥8 video</th></tr>{rows}</table>")


def user_report(uid, ev, sess, uf, hourly, assign, topic_lbl, vt, transitions, sensitivity):
    g = ev[ev.user_id == uid]
    f = uf[uf.user_id == uid].iloc[0]
    figs = []
    # 1. heatmap thứ x giờ
    m = g.groupby(["dow", "hour"]).size().unstack(fill_value=0).reindex(index=range(7), columns=range(24), fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 2.8))
    im = ax.imshow(m.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(0, 24, 2)); ax.set_yticks(range(7)); ax.set_yticklabels(F.DOW); ax.grid(False)
    ax.set_xlabel("Giờ trong ngày"); fig.colorbar(im, ax=ax, label="lượt xem")
    figs.append(("Khi nào xem? (thứ × giờ)", _png(fig)))
    # 2. theo tháng
    mo = g.groupby(g["ts"].dt.to_period("M")).size()
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.bar(mo.index.astype(str), mo.values, color=C1); ax.set_ylabel("lượt xem"); plt.xticks(rotation=45)
    figs.append(("Lượng xem theo tháng", _png(fig)))
    # 3. top kênh
    vc = g["channel_name"].dropna().value_counts().head(10)[::-1]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.barh([s[:40] for s in vc.index], vc.values, color=C1)
    figs.append(("Top 10 kênh", _png(fig)))
    # 4. thể loại hoặc chủ đề NLP
    topics_txt = ""
    if f.meta_coverage >= config.MIN_META_COVERAGE:
        cc = g["category_name"].dropna().value_counts(normalize=True).head(8)[::-1]
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.barh(cc.index, cc.values * 100, color=C2); ax.set_xlabel("% lượt xem")
        figs.append((f"Thể loại (YouTube API, phủ {_pct(f.meta_coverage)} video)", _png(fig)))
        d = g["length_bucket"].value_counts(normalize=True).reindex(g["length_bucket"].cat.categories).fillna(0)
        fig, ax = plt.subplots(figsize=(9, 2.4))
        ax.bar(d.index.astype(str), d.values * 100, color=C2); ax.set_ylabel("%")
        figs.append(("Độ dài video", _png(fig)))
    if vt is not None:
        t = g.merge(vt, on="video_id")["topic_id"].value_counts(normalize=True).head(8)
        lbl = topic_lbl.set_index("topic_id")["label"]
        topics_txt = "; ".join(f"[{lbl[i]}] {_pct(v)}" for i, v in t.head(3).items())
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.barh([lbl[i][:45] for i in t.index[::-1]], t.values[::-1] * 100, color="#10b981"); ax.set_xlabel("% lượt xem")
        figs.append(("Chủ đề tự phát hiện từ title (TF-IDF + KMeans)", _png(fig)))
        # Cùng một chủ đề có thể được xem ở các giờ khác nhau; đây là phân bố,
        # không phải bằng chứng về hoàn cảnh hay nguyên nhân xem.
        gt = g.merge(vt, on="video_id", how="inner")
        top_ids = gt["topic_id"].value_counts().head(5).index
        daypart_topic = (gt[gt["topic_id"].isin(top_ids)]
                           .groupby(["daypart", "topic_id"]).size()
                           .unstack(fill_value=0)
                           .reindex(["sang", "chieu", "toi", "dem"], fill_value=0))
        if len(daypart_topic):
            fig, ax = plt.subplots(figsize=(9, 3))
            daypart_topic.rename(columns=lambda i: _topic_name(i, topic_lbl)[:28]).plot(
                kind="bar", stacked=True, ax=ax, colormap="tab20")
            ax.set_xlabel("Khung giờ"); ax.set_ylabel("lượt xem"); ax.legend(fontsize=7, title="chủ đề")
            figs.append(("Chủ đề theo khung giờ", _png(fig)))
    # Chỉ xét hai video liền nhau trong cùng phiên và chỉ báo cáo chuyển chủ đề.
    ut = transitions[transitions["user_id"] == uid].head(8) if transitions is not None else pd.DataFrame()
    if len(ut):
        labels = [_transition_text(r, topic_lbl)[:65] for r in ut.iloc[::-1].itertuples()]
        fig, ax = plt.subplots(figsize=(9, max(2.4, 0.38 * len(ut) + 1.2)))
        ax.barh(labels, ut.iloc[::-1]["n_transitions"], color="#8b5cf6")
        ax.set_xlabel("số lần chuyển trong phiên")
        figs.append(("Chuyển dịch giữa các chủ đề", _png(fig)))
    # 5. phiên
    s = sess[sess.user_id == uid]
    fig, ax = plt.subplots(figsize=(9, 2.4))
    ax.hist(s["n_videos"].clip(upper=20), bins=range(1, 22), color=C1, align="left")
    ax.set_xlabel("số video / phiên (gộp ≥20)"); ax.set_ylabel("số phiên")
    figs.append(("Phân bố kích thước phiên xem", _png(fig)))

    cl = ""
    if assign is not None and uid in set(assign.user_id):
        cl = f'<p class="sub">Thuộc <b>{html.escape(assign.loc[assign.user_id == uid, "cluster_name"].iloc[0])}</b></p>'
    cards = "".join([_card(f"{f.n_events:,}", "lượt xem"), _card(f"{f.videos_per_active_day:.1f}", "video/ngày hoạt động"),
                     _card(f"{f.peak_hour}h", "giờ cao điểm"), _card(_pct(f.night_ratio), "xem ban đêm"),
                     _card(f"{f.avg_videos_per_session:.1f}", "video/phiên"), _card(_pct(f.channel_top5_share), "top 5 kênh")])
    rank = "".join(_rank_note(uf, uid, c, LABEL.get(c, c)) for c in
                   ["videos_per_active_day", "night_ratio", "avg_videos_per_session", "channel_top5_share", "repeat_ratio"])
    rank = f"<h2>So với nhóm</h2><table>{rank}</table>" if rank else ""
    body = "".join(f"<h2>{t}</h2>{img}" for t, img in figs)
    lis = "".join(f"<li>{x}</li>" for x in insights(f, topics_txt))
    page = f"""<!doctype html><html lang="vi"><meta charset="utf-8"><title>{uid}</title><style>{CSS}</style>
<h1>Hồ sơ xem YouTube: {html.escape(uid)}</h1>{cl}<div class="grid">{cards}</div>
<h2>Insight tự động</h2><ul>{lis}</ul>{rank}{_session_sensitivity_note(sensitivity, uid)}{body}
<div class="note">Lưu ý: lịch sử Takeout chỉ ghi <i>thời điểm mở video</i>, không cho biết xem bao lâu. "Phiên" và "thời lượng phiên" là xấp xỉ.
Dislike không lấy được công khai; chỉ số API (view/like) là giá trị hiện tại, không phải lúc người dùng xem.</div></html>"""
    config.REPORTS.mkdir(parents=True, exist_ok=True)
    (config.REPORTS / f"{uid}.html").write_text(page, encoding="utf-8")


def group_report(uf, hourly, daily, monthly, channels, categories, cl, topic_lbl, transitions, sensitivity,
                 topic_audit, comments=None, comment_sent=None):
    parts = []
    cols = ["user_id", "n_events", "videos_per_active_day", "night_ratio", "weekend_boost", "avg_videos_per_session",
            "channel_top5_share", "repeat_ratio", "shorts_ratio", "meta_coverage",
            "comment_positive_share", "comment_negative_share"]
    cols = [c for c in cols if c in uf.columns]
    t = uf[cols].copy()
    if cl:
        t = t.merge(cl["assign"][["user_id", "cluster_name"]], on="user_id")
    rows = "".join("<tr>" + "".join(f"<td>{(f'{v:.2f}' if isinstance(v, float) and not pd.isna(v) else '' if pd.isna(v) else v)}</td>"
                                    for v in r) + "</tr>" for r in t.itertuples(index=False))
    head = "".join(f"<th>{LABEL.get(c, c)}</th>" for c in t.columns)
    parts.append(f'<h2>Bảng tổng hợp {len(uf)} người dùng</h2><div style="overflow-x:auto"><table><tr>{head}</tr>{rows}</table></div>')
    parts.append("<h2>1. Hành vi theo thời gian</h2><p>Phân bố lượt mở video theo thời điểm; không suy ra thời lượng xem thực tế.</p>")
    if len(hourly):
        hg = hourly.groupby("hour")["n"].sum().reindex(range(24), fill_value=0)
        fig, ax = plt.subplots(figsize=(9, 2.5))
        ax.bar(hg.index, hg.values, color=C1); ax.set_xticks(range(0, 24, 2))
        ax.set_xlabel("giờ trong ngày"); ax.set_ylabel("lượt xem")
        parts.append(_png(fig))
    if len(daily):
        dg = daily.groupby("date")["n_views"].sum()
        fig, ax = plt.subplots(figsize=(9, 2.8))
        ax.plot(dg.index, dg.values, color=C1)
        ax.set_xlabel("Ngày"); ax.set_ylabel("Lượt xem"); fig.autofmt_xdate()
        parts.append(f"<h2>Nhịp xem theo ngày</h2>{_png(fig)}")
    if len(monthly):
        mg = monthly.groupby("month")["n_views"].sum()
        fig, ax = plt.subplots(figsize=(9, 2.6))
        ax.bar(mg.index, mg.values, color=C1); ax.set_ylabel("Lượt xem")
        plt.xticks(rotation=45)
        parts.append(f"<h2>Tháng có nhiều lượt xem nhất</h2>{_png(fig)}")
    parts.append("<h2>2. Sở thích nội dung</h2>")
    if len(channels):
        top = channels.head(15).iloc[::-1]
        fig, ax = plt.subplots(figsize=(9, 4.2))
        ax.barh(top["channel_name"].astype(str).str[:42], top["n_views"], color="#7c3aed")
        ax.set_xlabel("Lượt xem")
        parts.append(f"<h2>Kênh thường xuyên được xem</h2>{_png(fig)}")
        rows = "".join(f"<tr><td>{html.escape(str(r.channel_name))}</td><td>{r.n_views:,}</td><td>{r.n_users}</td><td>{_pct(r.share)}</td></tr>"
                       for r in channels.head(15).itertuples())
        parts.append(f"<table><tr><th>Kênh</th><th>Lượt xem</th><th>Số người</th><th>Tỷ trọng</th></tr>{rows}</table>")
    if len(categories):
        top = categories.head(12).iloc[::-1]
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.barh(top["category_name"], top["n_views"], color=C2)
        ax.set_xlabel("Lượt xem")
        parts.append(f"<h2>Thể loại chiếm nhiều thời gian xem (proxy theo lượt mở)</h2>{_png(fig)}")
    parts.append("<h2>3. Chuyển dịch giữa các chủ đề</h2>")
    if transitions is not None and len(transitions):
        gt = (transitions.groupby(["source_topic_id", "target_topic_id"])
                         .agg(n_transitions=("n_transitions", "sum"), n_users=("user_id", "nunique"))
                         .reset_index().sort_values("n_transitions", ascending=False).head(12))
        rows = "".join(
            f"<tr><td>{html.escape(_transition_text(r, topic_lbl))}</td><td>{r.n_transitions:,}</td><td>{r.n_users}</td></tr>"
            for r in gt.itertuples())
        parts.append("<p>Chỉ tính hai video mở liên tiếp trong cùng phiên; đây là liên hệ tuần tự, không chứng minh video trước gây ra video sau.</p>"
                     f"<table><tr><th>Chuyển chủ đề</th><th>Số lần</th><th>Số người</th></tr>{rows}</table>")
    else:
        parts.append('<div class="note">Chưa đủ title để gán chủ đề NLP, nên chưa thể tính chuyển dịch chủ đề một cách nhất quán.</div>')
    parts.append("<h2>4. Kiểu tiêu thụ nội dung</h2><p>Xem bảng tổng hợp phía trên để so sánh Shorts/video dài, xem lại, quy mô phiên và mức tập trung vào kênh.</p>")
    parts.append("<h2>5. Phân tích ngữ nghĩa</h2>")
    if topic_lbl is not None and len(topic_lbl):
        topic_rows = "".join(f"<tr><td>{html.escape(str(r.label))}</td><td>{r.n_videos:,}</td></tr>"
                             for r in topic_lbl.sort_values("n_videos", ascending=False).itertuples())
        parts.append("<p>Các cụm title được tạo bằng TF-IDF + KMeans; nhãn là từ khóa đại diện, không phải danh mục YouTube chính thức.</p>"
                     f"<table><tr><th>Chủ đề tiềm ẩn</th><th>Video duy nhất</th></tr>{topic_rows}</table>")
    else:
        parts.append('<div class="note">Cần ít nhất số title hợp lệ theo cấu hình để chạy topic modeling.</div>')
    if comments is not None and len(comments):
        dist = comment_sent["sentiment"].value_counts() if comment_sent is not None and len(comment_sent) else pd.Series(dtype=int)
        sentiment_note = " | ".join(f"{k}: {v:,}" for k, v in dist.items()) or "chưa phân loại"
        parts.append(f'<h2>Tín hiệu cảm xúc từ bình luận</h2><p>{sentiment_note}. '
                     'Đây là lexicon baseline, cần kiểm định thủ công trước khi dùng để gán nhãn người dùng.</p>')
    parts.append("<h2>6. Mẫu hình giữa các nhóm người xem</h2>")
    if cl:
        a = cl["assign"]
        fig, ax = plt.subplots(figsize=(8, 5.2))
        cmap = plt.get_cmap("tab10")
        for c, d in a.groupby("cluster"):
            ax.scatter(d.pc1, d.pc2, s=70, color=cmap(c), label=d.cluster_name.iloc[0])
            for r in d.itertuples():
                ax.annotate(r.user_id, (r.pc1, r.pc2), fontsize=7, xytext=(4, 4), textcoords="offset points")
        ax.legend(fontsize=7, loc="best"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        parts.append(f"<h2>Phân cụm người dùng (k={cl['k']}, silhouette={cl['silhouette']:.2f})</h2>{_png(fig)}")
        if cl["silhouette"] < 0.15:
            parts.append('<div class="note">Silhouette thấp: các cụm tách nhau chưa rõ, nên diễn giải như xu hướng chứ không phải "kiểu người" cố định.</div>')
        p = cl["profile"]
        fig, ax = plt.subplots(figsize=(9, 0.5 * len(p) + 2.4))
        im = ax.imshow(p.values, cmap="coolwarm", vmin=-2, vmax=2, aspect="auto"); ax.grid(False)
        ax.set_xticks(range(p.shape[1])); ax.set_xticklabels([LABEL[c] for c in p.columns], rotation=60, ha="right")
        ax.set_yticks(range(len(p))); ax.set_yticklabels([f"Cụm {i + 1}" for i in p.index]); fig.colorbar(im, label="z-score")
        parts.append(f"<h2>Đặc trưng từng cụm (z-score)</h2>{_png(fig)}")
        fig, ax = plt.subplots(figsize=(9, 3))
        hh = hourly.merge(a[["user_id", "cluster"]], on="user_id").groupby(["cluster", "hour"])["share"].mean().unstack(fill_value=0)
        for c, row in hh.iterrows():
            ax.plot(range(24), row.reindex(range(24), fill_value=0) * 100, color=cmap(c), label=f"Cụm {c + 1}")
        ax.set_xlabel("Giờ"); ax.set_ylabel("% lượt xem"); ax.legend(fontsize=7)
        parts.append(f"<h2>Nhịp xem theo giờ của từng cụm</h2>{_png(fig)}")
    else:
        parts.append(f'<div class="note">Mới có {len(uf)} người dùng; cần ≥{config.MIN_USERS_CLUSTER} để phân cụm.</div>')
    parts.append("<h2>Kiểm định chất lượng và độ tin cậy</h2>")
    distribution_cols = [c for c in ["videos_per_active_day", "night_ratio", "avg_videos_per_session", "channel_top5_share"] if c in uf]
    if distribution_cols and len(uf) > 1:
        fig, ax = plt.subplots(figsize=(9, 3.4))
        ax.boxplot([uf[c].dropna() for c in distribution_cols], tick_labels=[LABEL.get(c, c) for c in distribution_cols])
        ax.set_ylabel("giá trị theo từng người tham gia")
        ax.tick_params(axis="x", rotation=15)
        parts.append("<p>Phân bố dưới đây giữ lại khác biệt giữa người tham gia, thay vì chỉ dùng trung bình toàn nhóm.</p>" + _png(fig))
    if sensitivity is not None and len(sensitivity):
        sg = sensitivity.groupby("gap_minutes").agg(
            median_sessions_per_active_day=("sessions_per_active_day", "median"),
            median_videos_per_session=("avg_videos_per_session", "median"),
        ).reset_index()
        rows = "".join(f"<tr><td>{r.gap_minutes}</td><td>{r.median_sessions_per_active_day:.2f}</td>"
                       f"<td>{r.median_videos_per_session:.2f}</td></tr>" for r in sg.itertuples())
        parts.append("<p>Độ nhạy phiên: bảng dùng <b>trung vị giữa người tham gia</b>; báo cáo cá nhân cho biết mức thay đổi của từng người.</p>"
                     f"<table><tr><th>Ngưỡng (phút)</th><th>Trung vị phiên/ngày hoạt động</th><th>Trung vị video/phiên</th></tr>{rows}</table>")
    if topic_audit is not None and len(topic_audit):
        parts.append(f'<p>Topic model là gom cụm không giám sát, nên chưa báo cáo Precision/Recall/F1. '
                     f'Phiếu lấy mẫu <a href="topic_validation_sample.csv">{len(topic_audit)} title</a> đã được tạo để gán nhãn thủ công; '
                     'chỉ tính các chỉ số phân loại sau khi chốt taxonomy và ánh xạ cụm sang nhãn chuẩn.</p>')
    parts.append('<div class="note">So sánh cường độ xem dùng video/ngày hoạt động và tỷ lệ, không dùng tổng lượt mở thô khi thời gian quan sát khác nhau. '
                 'Hãy dùng mã người tham gia ẩn danh, loại bỏ thông tin định danh trước khi chia sẻ báo cáo và chỉ xử lý lịch sử khi có sự đồng ý. '
                 'Takeout chỉ ghi thời điểm <b>mở video</b>, không phải số phút đã xem; không suy luận tâm trạng, nghiện, hay tác động của thuật toán đề xuất từ báo cáo này.</div>')
    links = "".join(f'<li><a href="{u}.html">{u}</a></li>' for u in uf["user_id"])
    page = f"""<!doctype html><html lang="vi"><meta charset="utf-8"><title>Tổng hợp nhóm</title><style>{CSS}</style>
<h1>Tổng hợp nhóm người dùng YouTube</h1><p class="sub">Khảo sát khám phá trên {len(uf)} người dùng, {int(uf.n_events.sum()):,} lượt xem.</p>
<div class="note">Lượt xem là số lần mở video từ Takeout. Do không có thời lượng xem thực tế, "thời gian xem" trong dashboard được biểu diễn bằng lượt mở, độ dài video và thời lượng phiên xấp xỉ; không nên cộng trực tiếp thành phút đã xem.</div>
{''.join(parts)}<h2>Báo cáo từng người</h2><ul>{links}</ul></html>"""
    (config.REPORTS / "index.html").write_text(page, encoding="utf-8")


print('Các hàm tạo báo cáo HTML đã được nạp trong notebook.')
report = SimpleNamespace(user_report=user_report, group_report=group_report)


Các hàm tạo báo cáo HTML đã được nạp trong notebook.


In [5]:
import pandas as pd
from types import SimpleNamespace


def analyze(con):
    raw = store.read(con, "SELECT * FROM watch_raw")
    cnt = raw.groupby("user_id").size()
    small = cnt[cnt < config.MIN_EVENTS_USER].index.tolist()
    if small:
        print(f"  Bỏ qua user quá ít dữ liệu (<{config.MIN_EVENTS_USER} lượt): {len(small)} user")
        raw = raw[~raw.user_id.isin(small)]
    if raw.empty:
        print("  Chưa có dữ liệu để phân tích")
        return
    meta = store.read(con, "SELECT * FROM video_metadata")
    ev = features.build_events(raw, meta)
    sess = features.build_sessions(ev)
    session_features = features.session_clustering_features(ev, sess)
    session_clustering = cluster.run_sessions(session_features)
    if session_clustering:
        session_assignments = session_clustering["assign"]
        session_shares = cluster.session_cluster_shares(session_assignments)
    else:
        session_assignments = session_features.iloc[:0].copy()
        session_assignments["session_cluster"] = pd.Series(dtype="int64")
        session_assignments["pc1"] = pd.Series(dtype="float64")
        session_assignments["pc2"] = pd.Series(dtype="float64")
        session_shares = pd.DataFrame(columns=["user_id"])
    sensitivity = features.session_sensitivity(ev)
    uf = features.user_features(ev, sess)
    if len(session_shares):
        uf = uf.merge(session_shares, on="user_id", how="left")
    hourly = features.hourly_profile(ev)
    daily = features.daily_profile(ev)
    monthly = features.monthly_profile(ev)
    channels = features.channel_summary(ev)
    categories = features.category_summary(ev)
    vt, tl = nlp.fit_topics(raw)
    channel_meta = store.read(con, "SELECT * FROM channel_metadata")
    video_facts = features.build_video_facts(raw, meta, vt, channel_meta)
    if vt is not None:
        uf = uf.merge(nlp.topic_shares(ev, vt), on="user_id", how="left")
    transitions = features.topic_transitions(ev, vt)
    topic_audit = nlp.topic_validation_sample(raw, vt, tl)
    cl = cluster.run(uf)
    assign = cl["assign"] if cl else None
    store.replace_table(con, "user_features", uf)
    store.replace_table(con, "sessions", sess)
    store.replace_table(con, "session_clusters", session_assignments)
    store.replace_table(con, "session_sensitivity", sensitivity)
    store.replace_table(con, "user_hourly", hourly)
    store.replace_table(con, "daily_profile", daily)
    store.replace_table(con, "monthly_profile", monthly)
    store.replace_table(con, "channel_summary", channels)
    store.replace_table(con, "category_summary", categories)
    store.replace_table(con, "topic_transitions", transitions)
    # topic_id là mã nội bộ, có thể đổi khi mô hình được huấn luyện lại.
    video_topics = (vt[["video_id", "topic_id"]].copy() if vt is not None else
                    pd.DataFrame({"video_id": pd.Series(dtype="object"),
                                  "topic_id": pd.Series(dtype="int64")}))
    store.replace_table(con, "video_topics", video_topics)
    store.replace_table(con, "video_facts", video_facts)
    if assign is not None:
        store.replace_table(con, "user_clusters", assign)
    if tl is not None:
        store.replace_table(con, "topics", tl)
    store.replace_table(con, "topic_validation_sample", topic_audit)
    config.REPORTS.mkdir(parents=True, exist_ok=True)
    topic_audit.to_csv(config.REPORTS / "topic_validation_sample.csv", index=False, encoding="utf-8-sig")
    tables = set(con.execute("SHOW TABLES").fetchdf()["name"].tolist())
    comments = store.read(con, "SELECT * FROM video_comments") if "video_comments" in tables else None
    comment_sent = nlp.comment_sentiment(comments)
    if len(comment_sent):
        store.replace_table(con, "comment_sentiment", comment_sent)
        uf = uf.merge(nlp.user_comment_signals(ev, comment_sent), on="user_id", how="left")
    for uid in uf["user_id"]:
        user_report(uid, ev, sess, uf, hourly, assign, tl, vt, transitions, sensitivity)
    group_report(uf, hourly, daily, monthly, channels, categories, cl, tl, transitions, sensitivity,
                 topic_audit, comments, comment_sent)
    print(f"  Xong: {len(uf)} người dùng" + (f", {cl['k']} cụm (silhouette {cl['silhouette']:.2f})" if cl else ""))
    print(f"  Báo cáo: {config.REPORTS / 'index.html'}")


def run(enrich_api=True, max_api_calls=None, force=False):
    con = store.connect()
    try:
        print("[1-2] Ingestion")
        changed, errors = ingest.ingest_all(con)
        if errors:
            print(f"  Có {len(errors)} file cần xem lại; chi tiết nằm trong log ingestion.")
        print("[3] Enrichment")
        n_enriched = enrich.enrich_all(con, max_api_calls) if enrich_api else 0
        if enrich_api:
            enrich.enrich_comments(con)
        if not (changed or n_enriched or force):
            print("Không có dữ liệu mới; bật FORCE để chạy lại phân tích.")
            return
        print("[3-6] ETL -> Feature -> NLP -> Clustering -> Báo cáo")
        analyze(con)
        print(f"Database đã lưu: {config.DB_PATH}")
        print(f"Parquet đã xuất: {config.PARQUET}")
    finally:
        con.close()


def show_status():
    con = store.connect()
    try:
        summary = store.read(con, "SELECT user_id, count(*) AS luot_xem, min(watched_at) AS tu, max(watched_at) AS den FROM watch_raw GROUP BY 1 ORDER BY 1")
        print(summary.to_string(index=False) if len(summary) else "Chưa có watch history trong kho.")
        n = con.execute("SELECT count(DISTINCT video_id) FROM watch_raw").fetchone()[0]
        m = con.execute("SELECT count(*) FROM video_metadata WHERE status='ok'").fetchone()[0]
        print(f"\nVideo duy nhất: {n:,} | đã có metadata: {m:,} ({m / max(n, 1):.0%})")
    finally:
        con.close()


print('Pipeline và status đã được nạp; store/report được định nghĩa trong notebook này.')

Pipeline và status đã được nạp; store/report được định nghĩa trong notebook này.


## Kết quả

Các bảng DuckDB và Parquet được cập nhật theo lượt chạy. Báo cáo HTML được ghi vào `reports/`; danh sách title cần kiểm định thủ công ở `reports/topic_validation_sample.csv`.

## Hạn chế

Video không truy xuất được metadata vẫn có thể làm lệch mẫu. `view_count` là snapshot lúc lấy API, không phải lượt xem tại thời điểm người dùng xem. `user_id` chỉ được ẩn danh trong báo cáo. Báo cáo mô tả dữ liệu quan sát, không chứng minh quan hệ nhân quả.

In [6]:
if RUN_PIPELINE:
    run(enrich_api=ENRICH_API, max_api_calls=MAX_API_CALLS, force=FORCE)
else:
    print('Pipeline đang tắt; đặt RUN_PIPELINE=True để chạy.')

[1-2] Ingestion
[3] Enrichment
  101,219 video chưa có metadata (~2025 lượt gọi API, 2025 quota units)
    50/101,219
    1,050/101,219
    2,050/101,219
    3,050/101,219
    4,050/101,219
    5,050/101,219
    6,050/101,219
    7,050/101,219
    8,050/101,219
    9,050/101,219
    10,050/101,219
    11,050/101,219
    12,050/101,219
    13,050/101,219
    14,050/101,219
    15,050/101,219
    16,050/101,219
    17,050/101,219
    18,050/101,219
    19,050/101,219
    20,050/101,219
    21,050/101,219
    22,050/101,219
    23,050/101,219
    24,050/101,219
    25,050/101,219
    26,050/101,219
    27,050/101,219
    28,050/101,219
    29,050/101,219
    30,050/101,219
    31,050/101,219
    32,050/101,219
    33,050/101,219
    34,050/101,219
    35,050/101,219
    36,050/101,219
    37,050/101,219
    38,050/101,219
    39,050/101,219
    40,050/101,219
    41,050/101,219
    42,050/101,219
    43,050/101,219
    44,050/101,219
    45,050/101,219
    46,050/101,219
    47,050/101,21

C:\Users\miin\AppData\Local\Temp\ipykernel_25108\3578883549.py:27: UserWarning: Glyph 33853 (\N{CJK UNIFIED IDEOGRAPH-843D}) missing from font(s) DejaVu Sans.
  fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")


  Xong: 17 người dùng, 2 cụm (silhouette 0.21)
  Báo cáo: D:\project\bigdata\reports\index.html
Database đã lưu: D:\project\bigdata\data\warehouse.duckdb
Parquet đã xuất: D:\project\bigdata\data\parquet
